**This notebook is an exercise in the [Intro to Game AI and Reinforcement Learning](https://www.kaggle.com/learn/intro-to-game-ai-and-reinforcement-learning) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/play-the-game).**

---


# Introduction

You have seen how to define a random agent.  In this exercise, you'll make a few improvements.

To get started, run the code cell below to set up our feedback system.

In [ ]:
from learntools.core import binder
binder.bind(globals())
from learntools.game_ai.ex1 import *

### 1) A smarter agent

We can improve the performance without devising a complicated strategy, simply by selecting a winning move, if one is available.

In this exercise, you will create an agent that:
- selects the winning move, if it is available.  (_If there is more than one move that lets the agent win the game, the agent can select any of them._)
- Otherwise, it should select a random move.

To help you with this goal, we provide some helper functions in the code cell below. 

In [ ]:
import numpy as np

# Gets board at next step if agent drops piece in selected column
def drop_piece(grid, col, piece, config):
    next_grid = grid.copy()
    for row in range(config.rows-1, -1, -1):
        if next_grid[row][col] == 0:
            break
    next_grid[row][col] = piece
    return next_grid

# Returns True if dropping piece in column results in game win
def check_winning_move(obs, config, col, piece):
    # Convert the board to a 2D grid
    grid = np.asarray(obs.board).reshape(config.rows, config.columns)
    next_grid = drop_piece(grid, col, piece, config)
    # horizontal
    for row in range(config.rows):
        for col in range(config.columns-(config.inarow-1)):
            window = list(next_grid[row,col:col+config.inarow])
            if window.count(piece) == config.inarow:
                return True
    # vertical
    for row in range(config.rows-(config.inarow-1)):
        for col in range(config.columns):
            window = list(next_grid[row:row+config.inarow,col])
            if window.count(piece) == config.inarow:
                return True
    # positive diagonal
    for row in range(config.rows-(config.inarow-1)):
        for col in range(config.columns-(config.inarow-1)):
            window = list(next_grid[range(row, row+config.inarow), range(col, col+config.inarow)])
            if window.count(piece) == config.inarow:
                return True
    # negative diagonal
    for row in range(config.inarow-1, config.rows):
        for col in range(config.columns-(config.inarow-1)):
            window = list(next_grid[range(row, row-config.inarow, -1), range(col, col+config.inarow)])
            if window.count(piece) == config.inarow:
                return True
    return False

The `check_winning_move()` function takes four required arguments: the first two (`obs` and `config`) should be familiar, and: 
- `col` is any valid move 
- `piece` is either the agent's mark or the mark of its opponent.  

The function returns `True` if dropping the piece in the provided column wins the game (for either the agent or its opponent), and otherwise returns `False`.  To check if the agent can win in the next move, you should set `piece=obs.mark`.

**To complete this exercise, you need to define `agent_q1()` in the code cell below.  To do this, you're encouraged to use the `check_winning_move()` function.**  

The `drop_piece()` function (defined in the code cell above) is called in the `check_winning_move()` function.  Feel free to examine the details, but you won't need a detailed understanding to solve the exercise.

In [ ]:
import random

def agent_q1(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    # Take a win whenever one is on the board.
    for col in valid_moves:
        if check_winning_move(obs, config, col, obs.mark):
            return col
    return random.choice(valid_moves)

# Check your answer
q_1.check()


In [ ]:
# Lines below will give you a hint or solution code
#q_1.hint()
#q_1.solution()

### 2) An even smarter agent

In the previous question, you created an agent that selects winning moves.  In this problem, you'll amend the code to create an agent that can also block its opponent from winning.  In particular, your agent should:
- Select a winning move, if one is available.
- Otherwise, it selects a move to block the opponent from winning, if the opponent has a move that it can play in its next turn to win the game. 
- If neither the agent nor the opponent can win in their next moves, the agent selects a random move.

To help you with this exercise, you are encouraged to start with the agent from the previous exercise.  

**To check if the opponent has a winning move, you can use the `check_winning_move()` function, but you'll need to supply a different value for the `piece` argument.**  

In [ ]:
def agent_q2(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    # Win if we can.
    for col in valid_moves:
        if check_winning_move(obs, config, col, obs.mark):
            return col
    # Otherwise stop the opponent from winning next turn.
    for col in valid_moves:
        if check_winning_move(obs, config, col, obs.mark % 2 + 1):
            return col
    return random.choice(valid_moves)

# Check your answer
q_2.check()


In [ ]:
# Lines below will give you a hint or solution code
#q_2.hint()
#q_2.solution()

### 3) Looking ahead

So far, you have encoded an agent that always selects the winning move, if it's available.  And, it can also block the opponent from winning.

You might expect that this agent should perform quite well!  But how is it still possible that it can still lose the game?

In [ ]:
#q_3.hint()

In [ ]:
# Check your answer (Run this code cell to receive credit!)
q_3.solution()

### 4) Create your own agent

Amend the `my_agent()` function below to create your own agent.  Feel free to copy an agent that you created above.  

Note that you'll have to include all of the necessary imports and helper functions.  For an example of how this would look with the first agent you created in the exercise, take a look at **[this notebook](https://www.kaggle.com/alexisbcook/create-a-connectx-agent)**.

In [ ]:
def my_agent(obs, config):
    import random
    import time

    t_start = time.time()

    try:
        COLS = config.columns
        ROWS = config.rows
        K = config.inarow

        board = obs.board
        me = obs.mark
        valid = [c for c in range(COLS) if board[c] == 0]
        if not valid:
            return 0

        # --- bitboard layout -------------------------------------------
        # bit index = col * H + row, row 0 = bottom. H = ROWS + 1 leaves one
        # always-empty sentinel row per column, which is what stops vertical
        # and diagonal runs from wrapping between columns.
        H = ROWS + 1
        DIRS = (1, H, H + 1, H - 1)

        FULL = 0
        BOTTOM = 0
        COLMASK = []
        for c in range(COLS):
            cm = 0
            for r in range(ROWS):
                cm |= 1 << (c * H + r)
            COLMASK.append(cm)
            FULL |= cm
            BOTTOM |= 1 << (c * H)

        my_pos = 0
        mask = 0
        for c in range(COLS):
            for r in range(ROWS):
                cell = board[(ROWS - 1 - r) * COLS + c]
                if cell:
                    b = 1 << (c * H + r)
                    mask |= b
                    if cell == me:
                        my_pos |= b
        op_pos = mask ^ my_pos

        # Centre columns first: they sit in more winning lines, so they produce
        # cutoffs earlier.
        mid = (COLS - 1) / 2.0
        ORDER = sorted(range(COLS), key=lambda c: abs(c - mid))

        # Shift plans for "which empty squares complete a run of K".
        # For each direction and each position of the gap within the run, the
        # other K-1 cells must already be ours.
        PLANS = []
        for d in DIRS:
            for gap in range(K):
                PLANS.append(tuple((i - gap) * d for i in range(K) if i != gap))

        def shift(x, s):
            return x >> s if s >= 0 else x << -s

        def connected(pos):
            for d in DIRS:
                m = pos
                for i in range(1, K):
                    m &= pos >> (d * i)
                    if not m:
                        break
                if m:
                    return True
            return False

        def threat_squares(pos, msk):
            """Empty cells that would complete a K-run for `pos`."""
            res = 0
            for plan in PLANS:
                m = shift(pos, plan[0])
                for s in plan[1:]:
                    m &= shift(pos, s)
                    if not m:
                        break
                if m:
                    res |= m
            return res & FULL & ~msk

        CENTRE = COLMASK[COLS // 2]
        if COLS % 2 == 0:
            CENTRE |= COLMASK[COLS // 2 - 1]

        def evaluate(my, op, msk):
            my_t = threat_squares(my, msk)
            op_t = threat_squares(op, msk)
            playable = (msk + BOTTOM) & FULL
            score = 0
            score += 16 * bin(my_t).count("1") - 16 * bin(op_t).count("1")
            score += 40 * bin(my_t & playable).count("1")
            score -= 40 * bin(op_t & playable).count("1")
            score += 2 * bin(my & CENTRE).count("1")
            score -= 2 * bin(op & CENTRE).count("1")
            return score

        # --- search -----------------------------------------------------
        WIN = 10 ** 6
        INF = 10 ** 9
        budget = 0.90
        counter = [0]
        tt = {}

        class TimeUp(Exception):
            pass

        def negamax(my, op, msk, depth, alpha, beta, ply):
            counter[0] += 1
            if counter[0] & 511 == 0 and time.time() - t_start > budget:
                raise TimeUp

            nb = msk + BOTTOM
            moves = []
            for c in ORDER:
                b = nb & COLMASK[c]
                if b & FULL:
                    if connected(my | b):
                        return WIN - ply
                    moves.append(b)
            if not moves:
                return 0
            if depth == 0:
                return evaluate(my, op, msk)

            key = (my, msk)
            hit = tt.get(key)
            if hit is not None and hit[0] >= depth:
                _, flag, val = hit
                if flag == 0:
                    return val
                if flag == 1 and val > alpha:
                    alpha = val
                elif flag == 2 and val < beta:
                    beta = val
                if alpha >= beta:
                    return val

            alpha0 = alpha
            best = -INF
            for b in moves:
                v = -negamax(op, my | b, msk | b, depth - 1, -beta, -alpha, ply + 1)
                if v > best:
                    best = v
                if best > alpha:
                    alpha = best
                if alpha >= beta:
                    break

            if best <= alpha0:
                flag = 2
            elif best >= beta:
                flag = 1
            else:
                flag = 0
            tt[key] = (depth, flag, best)
            return best

        # --- root -------------------------------------------------------
        nb0 = mask + BOTTOM
        root_moves = []
        for c in ORDER:
            b = nb0 & COLMASK[c]
            if b & FULL:
                root_moves.append((c, b))

        # Win now, and never hand the opponent a win next move.
        for c, b in root_moves:
            if connected(my_pos | b):
                return c
        for c, b in root_moves:
            if connected(op_pos | b):
                return c

        best_move = root_moves[0][0]
        max_depth = ROWS * COLS - bin(mask).count("1")
        depth = 2
        while depth <= max_depth:
            try:
                alpha = -INF
                local_best = None
                for c, b in root_moves:
                    v = -negamax(op_pos, my_pos | b, mask | b, depth - 1, -INF, -alpha, 1)
                    if local_best is None or v > alpha:
                        alpha = v
                        local_best = c
                if local_best is not None:
                    best_move = local_best
                if alpha >= WIN - depth:
                    break
            except TimeUp:
                break
            if time.time() - t_start > budget * 0.5:
                break
            depth += 1

        return int(best_move)

    except Exception:
        try:
            return int(random.choice([c for c in range(config.columns) if obs.board[c] == 0]))
        except Exception:
            return 0


In [ ]:
# Run this code cell to get credit for creating an agent
q_4.check()

Run the next code cell to watch the agent play a game against the random agent.  You can re-run the code cell to play again!

In [ ]:
from kaggle_environments import evaluate, make

env = make("connectx", debug=True)
env.run([my_agent, "random"])
env.render(mode="ipython")

### 5) Submit to the competition

Now, it's time to make your first submission to the competition!  Run the next code cell to write your agent to a submission file.

In [ ]:
import inspect
import os

def write_agent_to_file(function, file):
    with open(file, "a" if os.path.exists(file) else "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(my_agent, "submission.py")

# Check that submission file was created
q_5.check()

Then, follow these steps:
1. Begin by clicking on the **Save Version** button in the top right corner of the window.  This will generate a pop-up window.  
2. Ensure that the **Save and Run All** option is selected, and then click on the **Save** button.
3. This generates a window in the bottom left corner of the notebook.  After it has finished running, click on the number to the right of the **Save Version** button.  This pulls up a list of versions on the right of the screen.  Click on the ellipsis **(...)** to the right of the most recent version, and select **Open in Viewer**.  This brings you into view mode of the same page. You will need to scroll down to get back to these instructions.
4. Click on the **Data** tab near the top of the screen.  Then, click on the file you would like to submit, and click on the **Submit** button to submit your results to the leaderboard.

You have now successfully submitted to the competition!

If you want to keep working to improve your performance, select the **Edit** button in the top right of the screen. Then you can change your code and repeat the process. There's a lot of room to improve, and you will climb up the leaderboard as you work.


Go to **"My Submissions"** to view your score and episodes being played.

# Keep going

Learn how to **[use heuristics](https://www.kaggle.com/alexisbcook/one-step-lookahead)** to improve your agent.

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intro-to-game-ai-and-reinforcement-learning/discussion) to chat with other learners.*